# Lily 1.5B Distillation v0.3 — Modal A100 40GB
**Offline SFT distillation of Lily-1.5B from Sarvam-105b-Distill-100k**

Single A100 40GB setup on Modal with inline execution, Unsloth, Flash Attention 2, native BFloat16 precision, step-by-step HF weight checkpointing, and lossless 16-bit model merging.

## Cell 1 — Install Dependencies

In [3]:
# unsloth[colab-new] auto-detects Ampere (A100) and installs Flash Attention 2.
# For Modal: bake this into your image definition for faster cold starts.
# Cell 1 — Install Dependencies (Modal-optimized with %uv)
%uv pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q
%uv pip install wandb -q
%uv pip install flash-attn --no-build-isolation -q
%uv pip install liger-kernel

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Using Python 3.12.6 environment at: /usr/local
Resolved 26 packages in 185ms
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
Prepared 1 package in 86ms
Installed 1 package in 118ms
 + liger-kernel==0.8.0
Note: you may need to restart the kernel to use updated packages.


## Cell 2 — Version Check & Hardware Probe

In [1]:
import unsloth  # MUST be first import — applies all patches to transformers/peft/trl
print(f"Unsloth : {unsloth.__version__}")

import torch, trl
print(f"TRL     : {trl.__version__}")
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.version.cuda}")

p = torch.cuda.get_device_properties(0)
print(f"\nGPU     : {p.name}")
print(f"VRAM    : {p.total_memory/1e9:.1f} GB")
print(f"Compute : cc={p.major}.{p.minor}")
print(f"BF16    : {'Supported' if p.major >= 8 else 'NOT supported (need cc>=8.0)'}")
print(f"FA2     : {'Supported' if p.major >= 8 else 'NOT supported (need cc>=8.0)'}")

assert torch.cuda.is_available(), "No GPU detected!"
assert p.major >= 8, f"Requires Ampere+ GPU (A100/H100/RTX3090+). Got cc={p.major}.{p.minor}"
print("\nHardware check passed")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth : 2026.5.2
TRL     : 0.22.2
PyTorch : 2.8.0+cu129
CUDA    : 12.9

GPU     : NVIDIA A100-SXM4-40GB
VRAM    : 42.4 GB
Compute : cc=8.0
BF16    : Supported
FA2     : Supported

Hardware check passed


## Cell 3 — Configuration

In [2]:
import os, torch

HF_USERNAME = "abhinav0231"

# Model
MODEL_NAME      = "abhinav0231/Lily-1.5b-v0.1"
MAX_SEQ_LENGTH  = 4096
LORA_RANK       = 32
LORA_ALPHA      = 64

# Dataset
DATASET_REPO    = "abhinav0231/Sarvam-105b-Distill-100k"
DATASET_CONFIG  = "chatml"   # pre-formatted <|im_start|>/<|im_end|> text column

# Training
# A100 40GB + QLoRA 4-bit: Lily 1.5B uses ~0.9GB model weights.
# Remaining ~39GB covers activations, optimizer states, and batch.
# Batch=4, GradAccum=4 => eff batch 16 (same as T4x2 DDP, but per single GPU).
# For even higher throughput, try BATCH_SIZE=8, GRAD_ACCUM=2.
NUM_EPOCHS      = 2
LEARNING_RATE   = 2e-5
WARMUP_STEPS    = 100
BATCH_SIZE      = 24
GRAD_ACCUM      = 1
SEED            = 42

# Checkpointing
SAVE_STEPS             = 500
CHECKPOINT_REPO        = f"{HF_USERNAME}/Lily-1.5b-distill-v3-checkpoints"
RESUME_FROM_CHECKPOINT = True

# Output repos
HF_ADAPTER_REPO = f"{HF_USERNAME}/Lily-1.5b-v0.3-adapter"
HF_MERGED_REPO  = f"{HF_USERNAME}/Lily-1.5b-v0.3"

# W&B
WANDB_PROJECT   = "superqwen"
WANDB_RUN_NAME  = "lily-1.5b-distill-v3-a100"

# Paths — on Modal use /root or a mounted Volume path
OUTPUT_DIR  = "/root/distill_output"
MERGED_DIR  = "/root/distill_merged"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MERGED_DIR, exist_ok=True)

# A100 = cc 8.0 => BF16 always available
DTYPE      = torch.bfloat16
MIXED_PREC = "bf16"

print("Config ready")
print(f"  Model       : {MODEL_NAME}")
print(f"  Dataset     : {DATASET_REPO} ({DATASET_CONFIG})")
print(f"  Epochs      : {NUM_EPOCHS}")
print(f"  Eff batch   : {BATCH_SIZE} x {GRAD_ACCUM} = {BATCH_SIZE * GRAD_ACCUM}  (single GPU)")
print(f"  LoRA r/a    : {LORA_RANK} / {LORA_ALPHA}")
print(f"  Dtype       : {DTYPE}  |  FA2: ON (auto via Unsloth)")
print(f"  Output      : {HF_MERGED_REPO}")
print(f"  Ckpt repo   : {CHECKPOINT_REPO}")

Config ready
  Model       : abhinav0231/Lily-1.5b-v0.1
  Dataset     : abhinav0231/Sarvam-105b-Distill-100k (chatml)
  Epochs      : 2
  Eff batch   : 24 x 1 = 24  (single GPU)
  LoRA r/a    : 32 / 64
  Dtype       : torch.bfloat16  |  FA2: ON (auto via Unsloth)
  Output      : abhinav0231/Lily-1.5b-v0.3
  Ckpt repo   : abhinav0231/Lily-1.5b-distill-v3-checkpoints


## Cell 4 — Secrets & Authentication

In [3]:
from huggingface_hub import login
import wandb

# On Modal: inject secrets via `modal secret create hf-secret HF_TOKEN=hf_...`
# and reference with `secrets=[modal.Secret.from_name("hf-secret")]` in @app.function.
# For ad-hoc notebook runs, set them as env vars or uncomment below:
HF_TOKEN    = os.environ.get("HF_TOKEN", "")
WANDB_TOKEN = os.environ.get("WANDB_API_KEY", "")

assert HF_TOKEN,    "Set HF_TOKEN (env var or Modal secret)"
assert WANDB_TOKEN, "Set WANDB_API_KEY (env var or Modal secret)"

login(token=HF_TOKEN, add_to_git_credential=False)
wandb.login(key=WANDB_TOKEN, relogin=True)

os.environ["HF_TOKEN"]                = HF_TOKEN
os.environ["WANDB_API_KEY"]           = WANDB_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"]  = HF_TOKEN
# Reduce VRAM fragmentation for long training runs
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:512"

print("Authenticated with HuggingFace & WandB")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: abhinav0231 (abhinav0231-krmangalam) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Authenticated with HuggingFace & WandB


## Cell 5 — Load Model & Apply LoRA (with FA2 + BF16)

**Key A100 differences from T4:**
- `dtype=torch.bfloat16` — BF16 has the same dynamic range as FP32 (8 exponent bits), making training more numerically stable than FP16. A100 has dedicated BF16 hardware units with the same throughput as FP16.
- **Flash Attention 2 is auto-enabled** by Unsloth when it detects cc>=8.0. You will see `FA2 = True` in the startup banner. No extra flag needed.
- Single GPU — no DDP considerations, no `ddp_broadcast_buffers` hacks.
- Still using QLoRA 4-bit even though A100 has headroom — keeps optimizer states small.

In [4]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = DTYPE,      # bfloat16 on A100
    load_in_4bit   = True,       # NF4 QLoRA: ~0.9GB for 1.5B, 39GB free for batches
    # FA2 is auto-detected and enabled by Unsloth on A100 (cc=8.0).
    # Startup banner will show: "FA [Xformers = None. FA2 = True]"
    # To explicitly disable: add  attn_implementation="eager"
)

tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "right"

# use_gradient_checkpointing="unsloth" = async CPU-pinned activation offloading
# Only ~1.9% speed penalty vs 20-30% for standard gradient checkpointing
model = FastLanguageModel.get_peft_model(
    model,
    r                          = LORA_RANK,
    lora_alpha                 = LORA_ALPHA,
    target_modules             = ["q_proj", "k_proj", "v_proj", "o_proj",
                                   "gate_proj", "up_proj", "down_proj"],
    lora_dropout               = 0,
    bias                       = "none",
    use_gradient_checkpointing = False,
    random_state               = SEED,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable params : {trainable/1e6:.1f}M / {total/1e6:.0f}M  ({100*trainable/total:.1f}%)")
print("Model ready: QLoRA 4-bit | BF16 | FA2 (auto-enabled)")

==((====))==  Unsloth 2026.5.2: Fast Qwen2 patching. Transformers: 4.56.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu129. CUDA: 8.0. CUDA Toolkit: 12.9. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/164 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/374 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Unsloth 2026.5.2 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Trainable params : 36.9M / 926M  (4.0%)
Model ready: QLoRA 4-bit | BF16 | FA2 (auto-enabled)


## Cell 6 — Load Dataset

In [5]:
from datasets import load_dataset

print(f"Loading: {DATASET_REPO}  [{DATASET_CONFIG}]")

dataset     = (
    load_dataset(DATASET_REPO, DATASET_CONFIG, split="train")
    .shuffle(seed=SEED)
)
val_dataset = load_dataset(DATASET_REPO, DATASET_CONFIG, split="validation")

print(f"  Train   : {len(dataset):,}")
print(f"  Val     : {len(val_dataset):,}")
print(f"  Columns : {dataset.column_names}")

Loading: abhinav0231/Sarvam-105b-Distill-100k  [chatml]


README.md: 0.00B [00:00, ?B/s]

chatml/train-00000-of-00002.parquet:   0%|          | 0.00/136M [00:00<?, ?B/s]

chatml/train-00001-of-00002.parquet:   0%|          | 0.00/135M [00:00<?, ?B/s]

chatml/validation-00000-of-00001.parquet:   0%|          | 0.00/5.67M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/91457 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1908 [00:00<?, ? examples/s]

  Train   : 91,457
  Val     : 1,908
  Columns : ['text']


## Cell 7 — W&B Init

In [6]:
import wandb

wandb.init(
    project  = WANDB_PROJECT,
    name     = WANDB_RUN_NAME,
    settings = wandb.Settings(init_timeout=300),
    config   = {
        "model"          : MODEL_NAME,
        "dataset"        : DATASET_REPO,
        "dataset_config" : DATASET_CONFIG,
        "lora_rank"      : LORA_RANK,
        "lora_alpha"     : LORA_ALPHA,
        "num_epochs"     : NUM_EPOCHS,
        "lr"             : LEARNING_RATE,
        "batch_size"     : BATCH_SIZE,
        "grad_accum"     : GRAD_ACCUM,
        "eff_batch"      : BATCH_SIZE * GRAD_ACCUM,
        "max_seq_length" : MAX_SEQ_LENGTH,
        "dtype"          : "bfloat16",
        "packing"        : True,
        "optimizer"      : "adamw_8bit",
        "flash_attn2"    : True,
        "gpu"            : "A100-40GB",
    },
)
print(f"W&B run: {WANDB_RUN_NAME}")

wandb: Tracking run with wandb version 0.26.1
wandb: Run data is saved locally in /root/wandb/run-20260511_050604-8hodmgw3
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run lily-1.5b-distill-v3-a100
wandb: ⭐️ View project at https://wandb.ai/abhinav0231-krmangalam/superqwen
wandb: 🚀 View run at https://wandb.ai/abhinav0231-krmangalam/superqwen/runs/8hodmgw3
wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai


W&B run: lily-1.5b-distill-v3-a100


## Cell 8 — Auto-Resume from HF Checkpoint

In [7]:
from huggingface_hub import HfApi, snapshot_download

_resume_ckpt = None

if RESUME_FROM_CHECKPOINT and CHECKPOINT_REPO:
    try:
        api   = HfApi()
        files = list(api.list_repo_files(CHECKPOINT_REPO, token=HF_TOKEN))
        nums  = set()
        for f in files:
            seg = f.split("/")[0]
            if seg.startswith("checkpoint-") and seg.split("-")[-1].isdigit():
                nums.add(int(seg.split("-")[-1]))
        if nums:
            latest    = max(nums)
            local_dir = os.path.join(OUTPUT_DIR, f"checkpoint-{latest}")
            print(f"Found checkpoint-{latest} — downloading ...")
            snapshot_download(
                repo_id        = CHECKPOINT_REPO,
                allow_patterns = [f"checkpoint-{latest}/*"],
                local_dir      = OUTPUT_DIR,
                token          = HF_TOKEN,
            )
            _resume_ckpt = local_dir
            print(f"Checkpoint downloaded -> {local_dir}")
        else:
            print("No checkpoints found — starting fresh.")
    except Exception as e:
        print(f"WARNING: Resume failed ({e}) — starting fresh.")

print(f"resume_from_checkpoint = {_resume_ckpt}")

No checkpoints found — starting fresh.
resume_from_checkpoint = None


## Cell 9 — Checkpoint Push Callback

In [8]:
from transformers import TrainerCallback
from huggingface_hub import upload_folder

class CheckpointPushCallback(TrainerCallback):
    """Push LoRA adapter checkpoint to HF Hub after every save."""

    def __init__(self, repo_id: str, token: str):
        self.repo_id = repo_id
        self.token   = token
        if repo_id:
            try:
                HfApi().create_repo(repo_id, token=token, exist_ok=True, private=True)
                print(f"Checkpoint repo ready: {repo_id}")
            except Exception as e:
                print(f"WARNING: could not create checkpoint repo: {e}")

    def on_save(self, args, state, control, **kwargs):
        if not self.repo_id:
            return
        step     = state.global_step
        ckpt_dir = os.path.join(args.output_dir, f"checkpoint-{step}")
        if not os.path.exists(ckpt_dir):
            return
        print(f"\nStep {step}: pushing checkpoint -> {self.repo_id} ...")
        try:
            upload_folder(
                folder_path     = ckpt_dir,
                repo_id         = self.repo_id,
                token           = self.token,
                path_in_repo    = f"checkpoint-{step}",
                commit_message  = f"Distill SFT checkpoint step {step}",
                ignore_patterns = ["*.lock"],
            )
            print(f"checkpoint-{step} pushed.")
        except Exception as e:
            print(f"WARNING: HF push failed at step {step}: {e}")

_callbacks = [CheckpointPushCallback(repo_id=CHECKPOINT_REPO, token=HF_TOKEN)]
print("CheckpointPushCallback ready")

Checkpoint repo ready: abhinav0231/Lily-1.5b-distill-v3-checkpoints
CheckpointPushCallback ready


## Cell 10 — SFTTrainer Config & Training

**Modal A100 SFTConfig Settings:**



In [10]:
import time
from trl import SFTConfig, SFTTrainer

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,

    num_train_epochs=NUM_EPOCHS,
    learning_rate=2e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,

    per_device_train_batch_size=24,
    gradient_accumulation_steps=1,

    bf16=True,
    fp16=False,

    optim="adamw_torch_fused",

    dataset_text_field="text",
    max_seq_length=4096,
    packing=True,
    dataset_num_proc=4,

    dataloader_num_workers=4,
    max_grad_norm=1.0,

    eval_strategy="steps",
    eval_steps=500,

    logging_strategy="steps",
    logging_steps=25,
    logging_first_step=True,

    save_strategy="steps",
    save_steps=300,
    save_total_limit=3,

    report_to=["wandb"],

    disable_tqdm=False,
    log_level="info",

    seed=SEED,
)

trainer = SFTTrainer(
    model            = model,
    args             = training_args,
    train_dataset    = dataset,
    eval_dataset     = val_dataset,
    processing_class = tokenizer,
    callbacks        = _callbacks,
)

print("=" * 62)
print(f"  SFT Distillation — {MODEL_NAME}")
print(f"  GPU     : A100 40GB  |  FA2={model.config._attn_implementation}  |  BF16=ON")
print(f"  Epochs  : {NUM_EPOCHS}  |  LR: {LEARNING_RATE}  | Warmup  : cosine | ratio={training_args.warmup_ratio}")
print(f"  Batch   : {BATCH_SIZE} x {GRAD_ACCUM} = {BATCH_SIZE * GRAD_ACCUM} (single GPU)")
print(f"  Opts    : QLoRA-4bit | FA2 | Liger | fused AdamW | packing")
print("=" * 62)
print()

t0 = time.time()
trainer.train(resume_from_checkpoint=_resume_ckpt)
elapsed = time.time() - t0
hrs, rem = divmod(int(elapsed), 3600); mins = rem // 60
print(f"\nTraining complete in {hrs}h {mins}m")

model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"LoRA adapter saved -> {OUTPUT_DIR}")
wandb.finish()

Unsloth: Tokenizing ["text"] (num_proc=4):   0%|          | 0/91457 [00:00<?, ? examples/s]

Unsloth: Packing train dataset (num_proc=4):   0%|          | 0/91457 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=4):   0%|          | 0/1908 [00:00<?, ? examples/s]

Unsloth: Packing eval dataset (num_proc=4):   0%|          | 0/1908 [00:00<?, ? examples/s]

[accelerate.utils.other|WARNING]Detected kernel version 4.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
Using auto half precision backend


...



Training completed. Do not forget to share your model on huggingface.co/models =)




checkpoint-2776 pushed.

Training complete in 5h 14m


loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--abhinav0231--Lily-1.5b-v0.1/snapshots/12c0254c5148e35c8867bcde2fcafaa69dd96a59/config.json
Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "dtype": "float16",
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 1536,
  "initializer_range": 0.02,
  "intermediate_size": 8960,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_atten

LoRA adapter saved -> /root/distill_output


wandb: uploading output.log; uploading wandb-summary.json; uploading config.yaml
wandb: uploading output.log; uploading config.yaml
wandb: 
wandb: Run history:
wandb:               eval/loss █▆▃▁▁
wandb:            eval/runtime ▄▁▂▂█
wandb: eval/samples_per_second ▄█▆▆▁
wandb:   eval/steps_per_second ▄█▆▆▁
wandb:             train/epoch ▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▆▆▇▇▇████
wandb:       train/global_step ▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇███
wandb:         train/grad_norm █▁▁▁▂▂▂▂▂▂▂▂▂▃▃▂▃▂▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃
wandb:     train/learning_rate ▃█████████▇▇▇▇▇▅▅▅▅▅▄▄▄▄▄▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁
wandb:              train/loss █▇▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb: 
wandb: Run summary:
wandb:               eval/loss 8.97307
wandb:            eval/runtime 540.6025
wandb: eval/samples_per_second 1.267
wandb:   eval/steps_per_second 0.318
wandb:              total_flos 2.1754504017676616e+18
wandb:             train/epoch 2
wandb:       train/global_step 2776
wandb:         train/grad_no

## Cell 11 — Merge LoRA Adapter & Push to Hub

In [11]:
from unsloth import FastLanguageModel

print("Loading base + adapter for merge ...")
# Load full precision BF16 for a lossless weight merge
model_m, tok_m = FastLanguageModel.from_pretrained(
    model_name     = OUTPUT_DIR,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = torch.bfloat16,  # BF16 merge on A100
    load_in_4bit   = False,           # full precision — no quantisation artefacts in merge
)

print("Merging LoRA weights into base model (bf16) ...")
model_m.save_pretrained_merged(MERGED_DIR, tok_m, save_method="merged_16bit")
print(f"Merged model saved locally -> {MERGED_DIR}")

print(f"Pushing to {HF_MERGED_REPO} ...")
model_m.push_to_hub_merged(HF_MERGED_REPO, tok_m, save_method="merged_16bit", token=HF_TOKEN)
print(f"Merged model -> https://huggingface.co/{HF_MERGED_REPO}")

Loading base + adapter for merge ...


loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--abhinav0231--Lily-1.5b-v0.1/snapshots/12c0254c5148e35c8867bcde2fcafaa69dd96a59/config.json
Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "dtype": "float16",
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 1536,
  "initializer_range": 0.02,
  "intermediate_size": 8960,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_atten

==((====))==  Unsloth 2026.5.2: Fast Qwen2 patching. Transformers: 4.56.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu129. CUDA: 8.0. CUDA Toolkit: 12.9. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--abhinav0231--Lily-1.5b-v0.1/snapshots/12c0254c5148e35c8867bcde2fcafaa69dd96a59/config.json
Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "dtype": "float16",
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 1536,
  "initializer_range": 0.02,
  "intermediate_size": 8960,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_atten

abhinav0231/Lily-1.5b-v0.1 does not have a padding token! Will use pad_token = <<|PAD_TOKEN|>>.
Merging LoRA weights into base model (bf16) ...


...

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

  ...31/Lily-1.5b-v0.3/model.safetensors:   0%|          | 21.6kB / 3.09GB            

Unsloth: Merging weights into 16bit: 100%|███████████████████████████████████████| 1/1 [01:16<00:00, 76.23s/it]


Unsloth: Merge process complete. Saved to `/root/abhinav0231/Lily-1.5b-v0.3`
Merged model -> https://huggingface.co/abhinav0231/Lily-1.5b-v0.3


## Cell 12 — Export GGUF (Q4_K_M, Q5_K_M, Q8_0, F16)

Run after Cell 11. Exports quantised GGUF files to `Lily-1.5b-v0.3-GGUF`.

In [14]:
from huggingface_hub import HfApi

GGUF_REPO = f"{HF_USERNAME}/Lily-1.5b-v0.3-GGUF"

print(f"Saving merged FP16 model locally...")
model_m.save_pretrained_merged(
    "merged_model",
    tok_m,
    save_method="merged_16bit",
)

print(f"Converting to GGUF locally...")
model_m.save_pretrained_gguf(
    "gguf_model",
    tok_m,
    quantization_method=["q4_k_m", "q5_k_m", "q8_0", "f16"],
)

print(f"Creating / uploading to HF repo -> {GGUF_REPO}")

api = HfApi(token=HF_TOKEN)

api.create_repo(
    repo_id=GGUF_REPO,
    repo_type="model",
    exist_ok=True,
)

api.upload_folder(
    folder_path="gguf_model",
    repo_id=GGUF_REPO,
    repo_type="model",
)

print(f"GGUF files uploaded -> https://huggingface.co/{GGUF_REPO}")

Saving merged FP16 model locally...


Configuration saved in merged_model/config.json


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...


Unsloth: Copying 1 files from cache to `merged_model`: 100%|█████████████████████| 1/1 [00:02<00:00,  2.93s/it]


Successfully copied all 1 files from cache to `merged_model`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


...

Unsloth: Copying 1 files from cache to `gguf_model`: 100%|███████████████████████| 1/1 [00:03<00:00,  3.59s/it]


Successfully copied all 1 files from cache to `gguf_model`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit: 100%|███████████████████████████████████████| 1/1 [00:25<00:00, 25.14s/it]


Unsloth: Merge process complete. Saved to `/root/gguf_model`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF bf16 might take 3 minutes.
\        /    [2] Converting GGUF bf16 to ['q4_k_m', 'q5_k_m', 'q8_0', 'f16'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Missing packages: libcurl4-openssl-dev
Unsloth: Will attempt to install missing system packages.
Unsloth: Installing packages: libcurl4-openssl-dev


RuntimeError: Unsloth: GGUF conversion failed: raw_input was called, but this frontend does not support input requests.